# Анализ и визуализация данных `check.csv`

## Цель ноутбука

Этот Jupyter Notebook одновременно является:

1. **аналитическим отчетом** по данным о заказах;
2. **практическим учебником по Matplotlib и Seaborn** для начинающих;
3. примером полного аналитического процесса: от загрузки и проверки данных до выводов и рекомендаций.

Для каждого графика мы отвечаем на четыре вопроса:

- **Что это за график?**
- **Что именно он показывает?**
- **Что можно увидеть в данных?**
- **Какие осторожные аналитические предположения можно сделать?**

> `SALES` в этом анализе рассчитывается как `QUANTITYORDERED × PRICEEACH`. Это расчетная сумма на уровне строки заказа, а не обязательно бухгалтерская выручка.

Matplotlib предоставляет интерфейс `pyplot` для построения и оформления графиков, а Seaborn ориентирован на статистические графики и тесно интегрирован с Matplotlib и pandas.

## 1. Логика аналитического процесса

Хороший аналитический отчет строится не от графика к данным, а наоборот:

**Бизнес-вопрос → проверка данных → подготовка → метрики → визуализация → интерпретация → выводы → рекомендации.**

В этом ноутбуке мы последовательно пройдем эти этапы.

Для группировки данных будем использовать `groupby`: это подход split–apply–combine — разбить данные на группы, посчитать агрегаты и собрать результат. 

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display, Markdown

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.titlesize"] = 15
plt.rcParams["axes.labelsize"] = 11

## 2. Загрузка данных

Сначала читаем CSV и знакомимся со структурой таблицы.

На этом этапе **не строим выводов о бизнесе**: сначала нужно понять размер данных, типы столбцов и качество заполнения.

In [ ]:
df_raw = pd.read_csv("data/check.csv")

print(f"Размер исходного набора: {df_raw.shape[0]} строк × {df_raw.shape[1]} столбцов")

audit = pd.DataFrame({
    "dtype": df_raw.dtypes.astype(str),
    "missing": df_raw.isna().sum(),
    "missing_%": (df_raw.isna().sum() / df_raw.shape[0] * 100).round(2),
    "unique": df_raw.nunique(dropna=True)
}).sort_values("missing_%", ascending=False)

display(audit)

In [ ]:
display(df_raw.head())

### Что проверяем

- **Количество строк** — сколько наблюдений у нас есть.
- **Количество столбцов** — какие признаки доступны.
- **Типы данных** — дата должна быть датой, числовые показатели — числами.
- **Пропуски** — могут влиять на агрегаты и графики.
- **Уникальные значения** — помогают понять категориальные признаки.

Пропуск не всегда означает ошибку. Например, отсутствие `STATE` может быть нормальным для стран, где этот атрибут не используется.

In [ ]:
df = df_raw.copy()

print('Количество пустых колонок ORDERDATE до преобразования типа', df["ORDERDATE"].isnull().sum())
df["ORDERDATE"] = pd.to_datetime(df["ORDERDATE"], errors="coerce")
print('Количество пустых колонок ORDERDATE после преобразования типа', df["ORDERDATE"].isnull().sum())

empty_rows = df.isna().all(axis=1).sum()
df = df.dropna(how="all").copy()

key_cols = ["ORDERNUMBER", "QUANTITYORDERED", "PRICEEACH", "ORDERDATE"]
before_key_filter = len(df)
df = df.dropna(subset=key_cols)

duplicate_count = df.duplicated().sum()

df["SALES"] = df["QUANTITYORDERED"] * df["PRICEEACH"]
df["YEAR"] = df["ORDERDATE"].dt.year
df["MONTH"] = df["ORDERDATE"].dt.month
df["MONTH_NAME"] = df["ORDERDATE"].dt.strftime("%b")
df["YEAR_MONTH"] = df["ORDERDATE"].dt.to_period("M").astype(str)

print(f"Полностью пустых строк удалено: {empty_rows}")
print(f"Строк до проверки ключевых полей: {before_key_filter}")
print(f"Строк после очистки ключевых полей: {len(df)}")
print(f"Полных дубликатов после очистки: {duplicate_count}")
print(f"Расчетные SALES: {df['SALES'].sum()}")

display(df.head())

## 3. Базовые показатели

Перед визуализацией полезно зафиксировать несколько опорных показателей.

Это помогает не потерять масштаб данных за красивыми графиками.

In [ ]:
agg_ser = pd.Series({
    "Количество строк": len(df),
    "Уникальные заказы": df["ORDERNUMBER"].nunique(),
    "Уникальные клиенты": df["CUSTOMERNAME"].nunique(),
    "Стран": df["COUNTRY"].nunique(),
    "Линий продуктов": df["PRODUCTLINE"].nunique(),
    "Расчетные продажи": df["SALES"].sum(),
    "Средние продажи на строку": df["SALES"].mean(),
    "Медианные продажи на строку": df["SALES"].median(),
    "Среднее количество единиц": df["QUANTITYORDERED"].mean(),
    "Средняя цена за единицу": df["PRICEEACH"].mean()
}).sort_values(ascending=False)
with pd.option_context('display.float_format', '{:.0f}'.format):
    display(agg_ser.to_frame("Значение"))

# 4. Визуализация: распределение количества товара

## График 1 — гистограмма

**Гистограмма** показывает, сколько наблюдений попадает в каждый диапазон числового показателя.

Здесь мы смотрим на `QUANTITYORDERED`.

### Как читать график

- ось X — диапазон количества заказанного товара;
- ось Y — число строк заказов;
- высота столбца — сколько наблюдений попало в соответствующий интервал.

Гистограмма полезна в начале анализа: распределение помогает увидеть диапазон, центральную тенденцию, асимметрию и потенциальные выбросы.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
sns.histplot(
    data=df, x="QUANTITYORDERED", 
    bins=20, kde=True, 
    palette="deep", legend=False, ax=ax
)
ax.set_title("Распределение количества единиц в строке заказа")
ax.set_xlabel("Количество единиц")
ax.set_ylabel("Количество строк")
plt.tight_layout()
plt.show()

q_mean = df["QUANTITYORDERED"].mean()
q_median = df["QUANTITYORDERED"].median()
q_min = df["QUANTITYORDERED"].min()
q_max = df["QUANTITYORDERED"].max()

display(Markdown(
    "**Интерпретация.** Количество единиц находится в диапазоне "
    f"**{q_min:.0f}–{q_max:.0f}**. Среднее значение — **{q_mean:.1f}**, "
    f"медиана — **{q_median:.1f}**.\n\n"
    "По форме гистограммы также можно оценить концентрацию заказов и частоту больших заказов.\n\n"
    "**Предположение:** экстремальные значения стоит отдельно проверить, прежде чем считать их аномалиями."
))

# 5. Продажи по продуктовым линиям

## График 2 — горизонтальная столбчатая диаграмма

Столбчатая диаграмма хорошо подходит для сравнения категорий.

Здесь каждая категория — `PRODUCTLINE`, а длина столбца — суммарный `SALES`.

График отвечает на вопрос: **какие продуктовые направления формируют наибольший объем продаж?**

In [ ]:
product_sales = (
    df.groupby("PRODUCTLINE", as_index=False)["SALES"]
      .sum()
      .sort_values("SALES", ascending=True)
)

fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(
    data=product_sales, x="SALES", y="PRODUCTLINE", 
    legend=False, ax=ax
)
ax.set_title("Расчетные продажи по продуктовым линиям")
ax.set_xlabel("SALES")
ax.set_ylabel("Продуктовая линия")
plt.tight_layout()
plt.show()

top_product = product_sales.iloc[-1]
bottom_product = product_sales.iloc[0]
total_sales = product_sales["SALES"].sum()

display(Markdown(
    f"**Интерпретация.** Лидер — **{top_product['PRODUCTLINE']}** с объемом "
    f"**{top_product['SALES']:,.0f}**. Минимальный объем у **{bottom_product['PRODUCTLINE']}** — "
    f"**{bottom_product['SALES']:,.0f}**. Доля лидера — примерно "
    f"**{top_product['SALES']/total_sales:.1%}**.\n\n"
    "**Предположение:** продуктовые линии имеют разный вклад в результат, поэтому лидеров и аутсайдеров "
    "целесообразно анализировать отдельно."
))

# 6. Динамика продаж во времени

## График 3 — линейный график

Линейный график используется, когда важна **динамика показателя во времени**.

Каждая точка — сумма `SALES` за месяц, а линия соединяет последовательные периоды.

Seaborn относит линейные графики к инструментам визуализации статистических взаимосвязей и особенно полезным для переменных, связанных со временем.

In [ ]:
monthly_sales = (
    df.groupby("YEAR_MONTH", as_index=False)["SALES"]
      .sum()
)

fig, ax = plt.subplots(figsize=(14, 6))
sns.lineplot(data=monthly_sales.assign(_ALL="Все"), x="YEAR_MONTH", y="SALES", marker="o", hue="_ALL", palette="deep", legend=False, ax=ax)
ax.set_title("Динамика расчетных продаж по месяцам")
ax.set_xlabel("Месяц")
ax.set_ylabel("SALES")
ax.tick_params(axis="x", rotation=60)
plt.tight_layout()
plt.show()

peak_month = monthly_sales.loc[monthly_sales["SALES"].idxmax()]
low_month = monthly_sales.loc[monthly_sales["SALES"].idxmin()]

display(Markdown(
    f"**Интерпретация.** Максимальный месячный объем — **{peak_month['YEAR_MONTH']}** "
    f"(**{peak_month['SALES']:,.0f}**), минимальный — **{low_month['YEAR_MONTH']}** "
    f"(**{low_month['SALES']:,.0f}**).\n\n"
    "Линия позволяет искать сезонность, пики и провалы. Один пик сам по себе еще не доказывает сезонность: "
    "для надежного вывода нужно сравнивать одинаковые месяцы разных лет.\n\n"
    "**Предположение:** есть периоды повышенного спроса; причины пиков следует проверить по календарю, "
    "промоакциям и другим бизнес-событиям."
))

# 7. Boxplot: продажи по размеру сделки

## График 4 — boxplot

**Ящик с усами (boxplot)** показывает распределение числового показателя внутри категорий.

Здесь сравниваются `SALES` для `Small`, `Medium` и `Large`.

Boxplot помогает увидеть медиану, центральные 50% наблюдений, разброс и потенциальные выбросы.

In [ ]:
order = ["Small", "Medium", "Large"]

fig, ax = plt.subplots(figsize=(11, 6))
sns.boxplot(
    data=df, x="DEALSIZE", y="SALES", order=order, 
    hue="DEALSIZE", palette="deep", legend=False, ax=ax
)
ax.set_title("Распределение расчетных продаж по размеру сделки")
ax.set_xlabel("Размер сделки")
ax.set_ylabel("SALES")
plt.tight_layout()
plt.show()

deal_summary = df.groupby("DEALSIZE")["SALES"].agg(["median", "mean", "count"]).reindex(order)
display(deal_summary)

display(Markdown(
    "**Интерпретация.** Boxplot позволяет сравнить не только средние значения, но и форму распределений. "
    "Если медиана и высота коробки заметно различаются между категориями, размер сделки связан с разным "
    "уровнем и разбросом продаж.\n\n"
    "**Предположение:** крупные сделки могут давать больше продаж на строку, но их редкость также важна. "
    "Сегмент нужно оценивать одновременно по объему, типичному значению и частоте."
))

# 8. Scatter plot: количество товара и цена

## График 5 — диаграмма рассеяния

**Scatter plot** показывает каждое наблюдение отдельной точкой.

- X — `QUANTITYORDERED`;
- Y — `PRICEEACH`;
- цвет — `DEALSIZE`.

Такой график помогает увидеть связь двух числовых переменных, группы и необычные наблюдения.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))
sns.scatterplot(
    data=df,
    x="QUANTITYORDERED",
    y="PRICEEACH",
    hue="DEALSIZE",
    alpha=0.65,
    ax=ax
)
ax.set_title("Связь количества заказанных единиц и цены")
ax.set_xlabel("Количество единиц")
ax.set_ylabel("Цена за единицу")
plt.tight_layout()
plt.show()

corr_q_price = df[["QUANTITYORDERED", "PRICEEACH"]].corr().iloc[0, 1]

display(Markdown(
    f"**Интерпретация.** Линейная корреляция между количеством и ценой составляет примерно "
    f"**{corr_q_price:.2f}**. Корреляция близка к 0 не означает отсутствие любой зависимости, "
    "а высокая корреляция не означает причинность.\n\n"
    "**Предположение:** облако точек можно использовать для поиска кластеров и необычных наблюдений; "
    "для проверки причин и статистической значимости нужен дополнительный анализ."
))

# 9. Корреляционная матрица

## График 6 — heatmap

**Тепловая карта корреляций** превращает таблицу коэффициентов корреляции в цветовую матрицу.

Она полезна для быстрого поиска пар числовых признаков, которые движутся вместе.

In [ ]:
numeric_cols = [
    "QUANTITYORDERED", "PRICEEACH", "ORDERLINENUMBER",
    "QTR_ID", "MONTH_ID", "YEAR_ID", "SALES"
]
corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="vlag", center=0, ax=ax)
ax.set_title("Корреляционная матрица числовых показателей")
plt.tight_layout()
plt.show()

corr_pairs = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool)).stack()
corr_pairs = corr_pairs.abs().sort_values(ascending=False)

display(Markdown(
    f"**Интерпретация.** Наиболее заметная пара по модулю корреляции: "
    f"**{corr_pairs.index[0][0]} ↔ {corr_pairs.index[0][1]}**, "
    f"модуль корреляции **{corr_pairs.iloc[0]:.2f}**.\n\n"
    "Это подсказка для дальнейшего исследования, а не доказательство причинности."
))

# 10. География продаж

## График 7 — Top-10 стран

Для большого числа категорий нельзя показывать все значения на одном графике — он становится нечитаемым.

Поэтому используем **Top-N**: оставляем 10 стран с максимальными продажами.

Это практический прием: сначала агрегировать данные, затем ограничивать число отображаемых категорий.

In [ ]:
country_sales = (
    df.groupby("COUNTRY", as_index=False)["SALES"]
      .sum()
      .sort_values("SALES", ascending=False)
      .head(10)
      .sort_values("SALES")
)

fig, ax = plt.subplots(figsize=(12, 7))
sns.barplot(
    data=country_sales, x="SALES", 
    y="COUNTRY", legend=False, ax=ax
)
ax.set_title("Топ-10 стран по расчетным продажам")
ax.set_xlabel("SALES")
ax.set_ylabel("Страна")
plt.tight_layout()
plt.show()

top_country = country_sales.iloc[-1]
display(Markdown(
    f"**Интерпретация.** Среди показанных стран лидирует **{top_country['COUNTRY']}** "
    f"с расчетными продажами **{top_country['SALES']:,.0f}**.\n\n"
    "**Предположение:** география продаж концентрируется в нескольких ключевых рынках. "
    "Это может быть основанием для сегментации маркетинговых и коммерческих решений."
))

# 11. Структура статусов заказов

## График 8 — столбчатое представление количества наблюдений в каждой категории.


Здесь он отвечает на вопрос: **сколько строк заказов относится к каждому статусу?**

Это не то же самое, что объем продаж: один статус может иметь много строк, но относительно небольшой денежный вклад.

In [ ]:
status_counts = df["STATUS"].value_counts().reset_index()
status_counts.columns = ["STATUS", "COUNT"]

fig, ax = plt.subplots(figsize=(11, 6))
sns.barplot(data=status_counts, x="COUNT", y="STATUS", legend=False, ax=ax)
ax.set_title("Количество строк по статусу заказа")
ax.set_xlabel("Количество строк")
ax.set_ylabel("Статус")
plt.tight_layout()
plt.show()

shipped_share = df["STATUS"].eq("Shipped").mean()
display(Markdown(
    f"**Интерпретация.** Статус **Shipped** составляет примерно **{shipped_share:.1%}** всех очищенных строк.\n\n"
    "Большая доля отгруженных заказов не исключает операционных проблем: для контроля процесса стоит отдельно "
    "изучать Cancelled, On Hold, Disputed и другие незавершенные статусы."
))

# 12. Годовой и квартальный анализ

## График 9 — группированная столбчатая диаграмма

Группировка по году и кварталу позволяет увидеть, **как менялась структура продаж внутри года**.

В отличие от общего годового графика, здесь можно сравнивать одинаковые кварталы разных лет.

In [ ]:
quarter_sales = (
    df.groupby(["YEAR", "QTR_ID"], as_index=False)["SALES"]
      .sum()
)

fig, ax = plt.subplots(figsize=(13, 6))
sns.barplot(data=quarter_sales, x="QTR_ID", y="SALES", hue="YEAR", palette="deep", ax=ax)
ax.set_title("Расчетные продажи по кварталам и годам")
ax.set_xlabel("Квартал")
ax.set_ylabel("SALES")
ax.legend(title="Год")
plt.tight_layout()
plt.show()

pivot_q = quarter_sales.pivot(index="YEAR", columns="QTR_ID", values="SALES")
display(pivot_q)

display(Markdown(
    "**Интерпретация.** Сравнение кварталов помогает отличить общий рост/снижение от сезонного эффекта. "
    "Если один и тот же квартал регулярно сильнее других, это может быть признаком сезонности. "
    "Неполный год необходимо трактовать осторожно."
))

# 13. Pareto-анализ клиентов

## График 10 — вклад клиентов + накопленная доля

Pareto-график помогает проверить принцип концентрации: **небольшая часть клиентов может формировать значительную часть результата**.

Сначала клиенты сортируются по `SALES`, затем рассчитывается накопленная доля продаж.

In [ ]:
customer_sales = (
    df.groupby("CUSTOMERNAME", as_index=False)["SALES"]
      .sum()
      .sort_values("SALES", ascending=False)
)

customer_sales["CUM_SHARE"] = customer_sales["SALES"].cumsum() / customer_sales["SALES"].sum()

top_n = min(20, len(customer_sales))
pareto = customer_sales.head(top_n).copy()

fig, ax1 = plt.subplots(figsize=(14, 7))
x = np.arange(len(pareto))

ax1.bar(x, pareto["SALES"])
ax1.set_xlabel("Клиенты (отсортированы по продажам)")
ax1.set_ylabel("SALES")
ax1.set_title("Pareto-анализ: концентрация расчетных продаж среди топ-клиентов")
ax1.set_xticks(x)
ax1.set_xticklabels(pareto["CUSTOMERNAME"], rotation=75, ha="right")

ax2 = ax1.twinx()
ax2.plot(x, pareto["CUM_SHARE"] * 100, marker="o")
ax2.set_ylabel("Накопленная доля, %")
ax2.set_ylim(0, 105)
ax2.axhline(80, linestyle="--", linewidth=1)

plt.tight_layout()
plt.show()

top10_share = customer_sales.head(10)["SALES"].sum() / customer_sales["SALES"].sum()
top20_share = customer_sales.head(min(20, len(customer_sales)))["SALES"].sum() / customer_sales["SALES"].sum()

display(Markdown(
    f"**Интерпретация.** Топ-10 клиентов формируют примерно **{top10_share:.1%}** расчетных продаж, "
    f"а топ-{min(20, len(customer_sales))} — около **{top20_share:.1%}**.\n\n"
    "Если концентрация высокая, бизнес может быть чувствителен к потере нескольких крупных клиентов. "
    "Это аргумент в пользу программ удержания ключевых клиентов и развития более широкого клиентского портфеля."
))

# 14. Мини-дашборд

Теперь объединим несколько основных метрик в одном компактном представлении.

Это пример перехода от отдельных визуализаций к **управленческому отчету**.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 11))

sns.barplot(
    data=product_sales.sort_values("SALES", ascending=False).assign(_ALL="Все"),
    x="SALES", y="PRODUCTLINE", hue="_ALL", palette="deep", legend=False, ax=axes[0, 0]
)
axes[0, 0].set_title("Продажи по продуктовым линиям")
axes[0, 0].set_xlabel("SALES")
axes[0, 0].set_ylabel("")

sns.lineplot(
    data=monthly_sales.assign(_ALL="Все"), x="YEAR_MONTH", y="SALES",
    marker="o", hue="_ALL", palette="deep", legend=False, ax=axes[0, 1]
)
axes[0, 1].set_title("Динамика продаж")
axes[0, 1].tick_params(axis="x", rotation=60)
axes[0, 1].set_xlabel("")
axes[0, 1].set_ylabel("SALES")

sns.boxplot(
    data=df, x="DEALSIZE", y="SALES",
    order=["Small", "Medium", "Large"], hue="DEALSIZE", palette="deep", legend=False, ax=axes[1, 0]
)
axes[1, 0].set_title("Распределение продаж по размеру сделки")
axes[1, 0].set_xlabel("Размер сделки")
axes[1, 0].set_ylabel("SALES")

sns.countplot(
    data=df, y="STATUS",
    order=df["STATUS"].value_counts().index,
    hue="STATUS", palette="deep", legend=False, ax=axes[1, 1]
)
axes[1, 1].set_title("Структура статусов")
axes[1, 1].set_xlabel("Количество строк")
axes[1, 1].set_ylabel("")

plt.tight_layout()
plt.show()

# 15. Построение аналитического отчета

## Шаг 1. Контекст и качество данных

В наборе присутствуют данные о заказах, клиентах, товарах, географии, статусах и датах.

После удаления полностью пустых строк и строк без ключевых полей мы работаем с очищенной выборкой. При этом некоторые поля (`ADDRESSLINE2`, `STATE`, `TERRITORY`) имеют значительное количество пропусков — это нужно учитывать при детальном географическом анализе.

## Шаг 2. Главная метрика

Используем:

`SALES = QUANTITYORDERED × PRICEEACH`

Перед использованием этой метрики в реальном управленческом отчете необходимо подтвердить ее бизнес-смысл и проверить наличие скидок, возвратов, налогов и других корректировок.

## Шаг 3. Ассортимент

График продуктовых линий показывает, какие направления дают основной объем продаж.

**Следующий вопрос:** почему лидер продается лучше — из-за количества заказов, цены, ассортимента или клиентского спроса?

## Шаг 4. Временная динамика

Месячный график позволяет обнаружить пики и провалы.

**Следующий вопрос:** повторяются ли пики ежегодно или они связаны с единичными событиями?

## Шаг 5. Клиенты

Pareto-анализ показывает, какую долю результата создают крупнейшие клиенты.

**Следующий вопрос:** насколько бизнес зависит от ограниченного числа клиентов?

## Шаг 6. Операционная часть

Статусы заказов показывают структуру процесса исполнения. Незавершенные и спорные статусы являются кандидатами для отдельного контроля.

## Шаг 7. Рекомендации для следующего этапа

1. Проверить причины пиков продаж по месяцам.
2. Сравнить продуктовые линии по количеству заказов, средней цене и среднему объему заказа.
3. Рассчитать продажи и маржинальность по клиентам.
4. Изучить отмененные и спорные заказы.
5. Проверить концентрацию продаж по клиентам и странам.
6. Построить RFM/cohort-анализ, если доступны необходимые даты.
7. Подтвердить формулу `SALES` с владельцем данных.

### Главный принцип

**График — это не вывод.** График делает закономерность видимой. Хороший аналитический отчет связывает визуальный паттерн с числом, контекстом, гипотезой и следующим аналитическим действием.